## How quickly does the system enter into a stressed regime

### Using a threshold based on 95% quantile of a baseline no-amortisation-peak-intensity (for each sectoral weighting scheme)

### Calculation of said threshold:

In [ ]:
import numpy as np
import scipy
from scipy.stats import ncx2
from scipy.special import hyp1f1
from scipy.optimize import brentq
from math import exp
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import kurtosis
from pathlib import Path

DECIMALS = 6

np.set_printoptions(precision=DECIMALS, suppress=True)
pd.options.display.float_format = lambda x: f"{x:.{DECIMALS}f}"
pd.set_option("display.precision", DECIMALS)

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman', 'Times New Roman', 'Times'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.figsize': (6.0, 4.0),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'lines.linewidth': 1.5,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.3,
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'text.usetex': False,
    'mathtext.fontset': 'cm',
})

IMG_DIR = Path("../images/first_passage_time")
IMG_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    'hist': '#5B8DB8',
    'kde': '#C0392B',
    'dark': '#2C3E50',
    'green': '#27AE60',
    'purple': '#8E44AD',
    'gold': '#D4AC0D',
    'scatter': '#5B8DB8',
}
MARKERS = ['o', 's', '^', 'D']

In [2]:
def lambda_max_generator(epsilon, y, theta, sigma, kappa, max_attempts=5, tol=1e-8):
    """
    
    Compute H^*_epsilon such that P_y(sigma_H < tau) = G_y(H; H) <= epsilon.
    This function implements Equation (11) using the closed-form in Equation (19).

    """

    def G_y(H):

        a = H / kappa  # Now using H as the Laplace exponent (since tau ~ Exp(H))
        b = 2 * kappa * theta / sigma**2
        z_y = 2 * kappa * y / sigma**2
        z_H = 2 * kappa * H / sigma**2

        num = hyp1f1(a, b, z_y)
        denom = hyp1f1(a, b, z_H)
        
        return num / denom  # This is G_y(H; H) = E[e^{-H * sigma_H}]

    def root_function(H):
        return G_y(H) - epsilon

    # Bracketing interval
    H_min = y + 1e-8
    H_max = y + 10.0

    # Expand H_max until G_y(H_max) < epsilon
    for _ in range(max_attempts):
        try:
          H_star = scipy.optimize.toms748(root_function, H_min, H_max, xtol=tol)
          
          if root_function(H_star)<=0:
            
            return H_star
          
          else:
          
            while root_function(H_star)>0:
              H_star = H_star + 1e-2
          
            return H_star
          
        except Exception:
            pass
        
        H_max += 5.0

    # Fallback: scan manually to find conservative H
    print("[Warning] brentq failed to converge. Using fallback grid search.")
    H_vals = np.linspace(H_min, H_max + 50, 1000)
    for H in H_vals:
        if G_y(H) <= epsilon:
            return H

    raise RuntimeError("Unable to find H^*_epsilon. Try expanding search space.")


In [3]:
def cir_transition_sample_per_sector(y_vec, tau, kappa_vec, theta_vec, sigma_vec, rng):
    """
    y_vec, kappa_vec, theta_vec, sigma_vec are arrays of length J (sectors).
    Returns xi_vec: sampled Y_{t+tau} per sector (length J).
    """
    if tau <= 0:
        return y_vec.copy()
    
    one_minus = -np.expm1(-kappa_vec * tau)  # = 1 - exp(-kappa*tau)
    # avoid zeros
    one_minus = np.where(one_minus <= 0, 1e-16, one_minus)
    
    c = (sigma_vec * sigma_vec * one_minus) / (4.0 * kappa_vec)
    d = 4.0 * kappa_vec * theta_vec / (sigma_vec * sigma_vec)
    
    # noncentrality parameters
    nc = (4.0 * kappa_vec * np.exp(-kappa_vec * tau) * y_vec) / (sigma_vec * sigma_vec * one_minus)
    
    # guard
    d = np.maximum(d, 1e-12)
    nc = np.maximum(nc, 0.0)
    
    # sample per sector (simple looping works fine since J would be typically small)
    xi = np.empty_like(y_vec, dtype=float)

    for j in range(len(y_vec)):
    
        # If df or nc are extreme, ncx2.rvs might throw an error
        try:
            Z = ncx2.rvs(df=d[j], nc=nc[j], random_state=rng)
    
        except Exception:
            Z = ncx2.rvs(df=max(d[j],1e-6), nc=0.0, random_state=rng) + nc[j]
    
        xi[j] = c[j] * Z
    
    xi = np.maximum(xi, 0.0)
    
    return xi


In [4]:
LGD_MEAN = 0.45
LGD_STD = 0.15

def get_beta_params(mu, sigma):
    if sigma**2 >= mu * (1 - mu):
        raise ValueError("Standard deviation is too high for this mean.")

    nu = (mu * (1 - mu) / sigma**2) - 1
    alpha = mu * nu
    beta = (1 - mu) * nu
    return alpha, beta

alpha_lgd, beta_lgd = get_beta_params(LGD_MEAN, LGD_STD)

In [5]:
from scipy.interpolate import interp1d

def loss_distribution_plot(Payoff_T, index, tag=""):
    counts, bin_edges = np.histogram(Payoff_T, bins=10, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    f_interp = interp1d(bin_centers, counts, kind='cubic', fill_value="extrapolate")
    x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), 500)
    y_smooth = f_interp(x_smooth)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(Payoff_T, bins=100, density=True, alpha=0.45, color=COLORS['hist'],
            edgecolor='white', linewidth=0.3, label='Histogram')
    ax.plot(x_smooth, y_smooth, color=COLORS['kde'], linewidth=1.8, label='KDE')
    ax.set_xlabel('Loss')
    ax.set_ylabel('Density')

    fname = f"loss_dist_{tag}" if tag else f"loss_dist_{index}"
    for fmt in ('pdf', 'png'):
        fig.savefig(IMG_DIR / f"{fname}.{fmt}", format=fmt)
    plt.close(fig)

In [6]:
def expected_shortfall(losses, alpha=0.95):
    var_alpha = np.quantile(losses, alpha)
    tail_losses = losses[losses >= var_alpha]
    return float(np.round(tail_losses.mean(), DECIMALS))

def compute_loss_metrics(losses, alpha=0.95):
    var_alpha = np.quantile(losses, alpha)
    es = expected_shortfall(losses, alpha)

    # E_max_intensity = np.mean(Y_max_vals)

    metrics = {
        # "Mean Max Intensity": E_max_intensity,
        "Mean": np.mean(losses),
        "Std": np.std(losses),
        "VaR": var_alpha,
        "ES": es,
        "Excess ES": es - var_alpha,
        "ExcessKurtosis": kurtosis(losses)
    }
    return {key: float(np.round(value, DECIMALS)) for key, value in metrics.items()}

In [7]:
loan_types = ["bullet", "linear", "french", "negative"]

def bullet_exposure(contract, T, t):
    P = float(contract["P"])
    return P if t < T else 0.0


def linear_exposure(contract, T, t):
    P = float(contract["P"])
    periods_per_year = float(contract["N"])
    if periods_per_year <= 0 or T <= 0:
        return 0.0

    period = 1.0 / periods_per_year  # years per period
    total_periods = int(np.ceil(T * periods_per_year))
    payments_made = int(np.floor(t / period))
    payments_made = int(np.clip(payments_made, 0, total_periods))
    principal_paid = (payments_made / total_periods) * P
    return max(P - principal_paid, 0.0)


def french_exposure(contract, T, t):
    P = float(contract["P"])
    periods_per_year = float(contract["N"])
    r_annual = float(contract["r"])
    if periods_per_year <= 0 or T <= 0:
        return 0.0

    period = 1.0 / periods_per_year  # years per period
    total_periods = int(np.ceil(T * periods_per_year))
    payments_made = int(np.floor(t / period))
    payments_made = int(np.clip(payments_made, 0, total_periods))
    periods_remaining = total_periods - payments_made
    if periods_remaining <= 0:
        return 0.0

    r_period = r_annual / periods_per_year  # annual to per-period rate

    if r_period == 0.0:
        payment = P / total_periods
        return payment * periods_remaining

    annuity_payment = P * r_period / (1.0 - (1.0 + r_period) ** (-total_periods))
    pv_remaining = annuity_payment * (1.0 - (1.0 + r_period) ** (-periods_remaining)) / r_period
    return pv_remaining


def negative_amortization_exposure(contract, T, t):
    P = float(contract["P"])
    periods_per_year = float(contract["N"])
    r_annual = float(contract.get("r", 0.0))
    if periods_per_year <= 0 or T <= 0:
        return max(P, 0.0)

    period = 1.0 / periods_per_year  # years per period
    total_periods = int(np.ceil(T * periods_per_year))
    periods_elapsed = int(np.floor(t / period))
    periods_elapsed = int(np.clip(periods_elapsed, 0, total_periods))

    r_period = r_annual / periods_per_year  # annual to per-period rate

    exposure = P * ((1.0 + r_period) ** periods_elapsed)

    return exposure if t < T else 0.0


def exposure_at_time(contract, T, t):
    loan_type = str(contract["type"]).lower()
    dispatch = {
        "bullet": bullet_exposure,
        "linear": linear_exposure,
        "french": french_exposure,
        "negative": negative_amortization_exposure,
    }
    if loan_type not in dispatch:
        raise ValueError(f"Unknown loan type: {loan_type}")
    return dispatch[loan_type](contract, T, t)


In [8]:
def generate_portfolio_weights(N, J, composition_type, concentration_params=None, rng=None):
    """
    Build portfolio weights under three stylised configurations plus a random fallback.

    - concentrated: one dominant sector for every obligor (eg for J=3 [0.7, 0.2, 0.1])
    - balanced: evenly distributed across sectors (eg for J=3  [1/3, 1/3, 1/3])
    - mixed: partial concentration with residual diversified weights (eg for J=3  [0.5, 0.3, 0.2])
    - random: symmetric Dirichlet for variation.
    """
    if rng is None:
        rng = np.random.default_rng()

    if concentration_params is None:
        concentration_params = {}

    noise_level = concentration_params.get("noise_level", 0.0)
    alpha_random = concentration_params.get("alpha", 1.0)

    if composition_type == "concentrated":
        pattern = _pattern_concentrated(J)
        W = _apply_noise_and_normalize(pattern, N, noise_level, rng)
    elif composition_type == "balanced":
        pattern = _pattern_balanced(J)
        W = _apply_noise_and_normalize(pattern, N, noise_level, rng)
    elif composition_type == "mixed":
        pattern = _pattern_mixed(J)
        W = _apply_noise_and_normalize(pattern, N, noise_level, rng)
    elif composition_type == "random":
        W = _generate_random_weights(N, J, {"alpha": alpha_random}, rng)
    else:
        raise ValueError(f"Unknown composition_type: {composition_type}")

    return W


def _pattern_concentrated(J, dominant=0.7):
    if J <= 0:
        return np.array([])
    if J == 1:
        return np.array([1.0])
    residual = max(1.0 - dominant, 0.0)
    tail = residual / (J - 1)
    vec = np.full(J, tail)
    vec[0] = dominant
    return vec


def _pattern_balanced(J):
    if J <= 0:
        return np.array([])
    return np.full(J, 1.0 / J)


def _pattern_mixed(J, primary=0.5, secondary=0.3):
    if J <= 0:
        return np.array([])
    vec = np.zeros(J)
    vec[0] = min(primary, 1.0)
    if J >= 2:
        vec[1] = min(secondary, max(1.0 - vec[0], 0.0))
    residual = max(1.0 - vec.sum(), 0.0)
    if J > 2:
        vec[2:] = residual / (J - 2)
    elif J == 1:
        vec[0] = 1.0
    else:
        vec[1] += residual
    return vec


def _apply_noise_and_normalize(pattern, N, noise_level, rng):
    base = np.tile(pattern, (N, 1))
    if noise_level > 0:
        noise = rng.normal(0.0, noise_level, size=base.shape)
        base = base + noise
    base = np.maximum(base, 1e-6)
    base = base / base.sum(axis=1, keepdims=True)
    return base


def _generate_random_weights(N, J, params, rng):
    alpha = params.get("alpha", 1.0)
    W = np.zeros((N, J))
    for i in range(N):
        W[i] = rng.dirichlet(np.full(J, alpha))
    return W

In [9]:
def plot_Ymax_vs_loss(Y_max_vals, losses, tag=""):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(Y_max_vals, losses, alpha=0.35, s=12, color=COLORS['scatter'], edgecolors='none')
    ax.set_xlabel(r"Peak Systemic Intensity $Y_{\max}$")
    ax.set_ylabel("Total Portfolio Loss")

    if tag:
        for fmt in ('pdf', 'png'):
            fig.savefig(IMG_DIR / f"ymax_vs_loss_{tag}.{fmt}", format=fmt)
    plt.close(fig)

In [10]:
def simulate_sector_cir_model(kappa, theta, sigma, T, delta, W, eps, lambda_benchmark,
                              Nfirms, idiosyncratic_factor, loan_contracts, alpha_lgd, beta_lgd, rng=None):

    import warnings

    if rng is None:
        rng = np.random.default_rng()

    # ensure shapes
    J = len(theta)
    assert W.shape == (Nfirms, J)
    idiosyncratic_factor = np.asarray(idiosyncratic_factor).reshape(Nfirms,)
    lambda_benchmark = np.asarray(lambda_benchmark).reshape(J,)

    delta = np.asarray(delta)
    if delta.ndim == 1:
        delta = np.tile(delta.reshape(1, J), (Nfirms, 1))
    assert delta.shape == (Nfirms, J)

    # init
    t = 0.0
    Y_t = theta.copy().astype(float)   # sectoral intensities (J-vector), initialised at theta
    events = []
    owed = []
    marks = []
    defaulter_idio_factor = []
    sector_contributions = []
    alive = np.arange(Nfirms, dtype=int)

    # precompute some arrays for speed
    kappa = np.asarray(kappa, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    theta = np.asarray(theta, dtype=float)

    max_Y = np.sum(Y_t)

    # event loop
    while (t < T) and (alive.size > 0):

        lambda_max = np.maximum(Y_t, lambda_benchmark)  # J-vector
        # Hepsilon = sum_i [ X_i + w_i · lambda_max ] over alive firms
        
        # We can compute per-firm value and sum
        per_firm_sys = W[alive].dot(lambda_max)   # length alive.size
        per_firm_idio = idiosyncratic_factor[alive]
        Hepsilon = per_firm_sys.sum() + per_firm_idio.sum()

        if Hepsilon <= 0:
            break

        # sample a candidate waiting time
        tau = rng.exponential(1.0 / Hepsilon)
        t_candidate = t + tau
        if t_candidate >= T:
            break

        # sample sectoral Y at t_candidate conditional on no defaults in the waiting time
        Y_proposed = cir_transition_sample_per_sector(Y_t, tau, kappa, theta, sigma, rng)

        # compute proposed per-firm intensities (scalar values) using Y_proposed
        per_firm_sys_prop = W[alive].dot(Y_proposed)
        per_firm_total_prop = per_firm_sys_prop + per_firm_idio
        lambda_proposed_total = per_firm_total_prop.sum()

        # acceptance probability: Xi / Hepsilon where Xi = lambda_proposed_total
        accept_prob = min(lambda_proposed_total / Hepsilon, 1.0)
        if rng.random() < accept_prob:
            # accept a default at t_candidate
            t = t_candidate
            events.append(t)

            # choose which alive firm defaulted
            probs = per_firm_total_prop / lambda_proposed_total
            probs = np.maximum(probs, 0.0)
            probs = probs / probs.sum()
            selected_local_idx = rng.choice(len(alive), p=probs)
            selected_firm = int(alive[selected_local_idx])

            # record idiosyncratic factor
            defaulter_idio_factor.append(float(idiosyncratic_factor[selected_firm]))

            sector_contributions.append(W[selected_firm].copy())

            # mark (loss) from contract exposure
            contract = loan_contracts[selected_firm]
            exposure = exposure_at_time(contract, T, t)
            if exposure < 0:
                warnings.warn("Exposure computed negative; clamping to zero.")
                exposure = max(exposure, 0.0)
            if t >= T and exposure > 0:
                warnings.warn("Exposure positive after maturity; forcing to zero.")
                exposure = 0.0

            owed.append(exposure)
            
            LGD = rng.beta(alpha_lgd, beta_lgd)
            mark = LGD * exposure
            # mark =  exposure
            if mark < 0:
                warnings.warn("Loss mark negative; clamping to zero.")
                mark = max(mark, 0.0)
            marks.append(mark)

            # update Y_t with contagion
            Y_t = Y_proposed + delta[selected_firm] * mark/contract["P"]

            # remove defaulted firm from alive set
            alive = np.delete(alive, selected_local_idx)

        else:
            # reject, ie no default: advance time and set Y_t = Y_proposed
            t = t_candidate
            Y_t = Y_proposed
        
        max_Y = max(max_Y, np.sum(Y_t))

    return (np.array(events), np.array(marks), np.array(owed),
            np.array(defaulter_idio_factor), np.array(sector_contributions), max_Y)

In [11]:
def run_simulation(params, loan_contracts, num_trials, seed=42):

    rng = np.random.default_rng(seed)
    eps = 1e-5
    losses = np.zeros(num_trials)
    Y_max_vals = np.zeros(num_trials)
    num_defaults = np.zeros(num_trials, dtype=int)

    for i in range(num_trials):
        events, marks, _, _, _, max_Y = simulate_sector_cir_model(params['kappa'], params['theta'], params['sigma'], 
                          params['T'], params['delta'], params['W'], 1e-5, 
                          params['lambda_benchmark'], 
                          params['Firms'], params['idio_factor'], 
                          loan_contracts,
                          alpha_lgd, beta_lgd,
                          rng)
        losses[i] = np.sum(marks)
        Y_max_vals[i] = max_Y
        num_defaults[i] = len(events)

    return losses, Y_max_vals, num_defaults

## Balanced sectoral config; Lower contagion (0.01)

In [12]:
rng = np.random.default_rng(1265883927568217)

N_firms = 1000
J = 3

c = 0.01

T = 2.0

NUM_TRIALS = 2500

kappa = rng.uniform(0.5, 1.5, J)
theta = rng.uniform(0.001, 0.051, J)
sigma_inter = rng.uniform(0.0, 0.2, J)
sigma = np.minimum(np.sqrt(2*kappa*theta), sigma_inter)

idio_factor = rng.uniform(0.01, 0.03, N_firms)

feller = (2 * kappa * theta) / (sigma**2)
print("Feller condition (should be >=1):", feller)

W = generate_portfolio_weights(N_firms, J, "balanced", rng=rng)

delta = c * W

lambda_benchmark = np.array([
    lambda_max_generator(1e-4, theta[j], theta[j],
                         sigma[j], kappa[j])
    for j in range(J)
])


params = {
    'Firms': N_firms,
    'Sectors': J,
    'Global Senstivity Param': c,
    'kappa': kappa,
    'theta': theta,
    'sigma': sigma,
    'delta': delta,
    'lambda_benchmark': lambda_benchmark,
    'idio_factor': idio_factor,
    'W': W,
    'T': T
}

Feller condition (should be >=1): [3.637663 4.700201 5.508869]


In [13]:
params_df = pd.DataFrame({
    'kappa': kappa,
    'theta': theta,
    'sigma': sigma,
    'lambda_benchmark': lambda_benchmark
}, index=[f'Sector_{j}' for j in range(J)])

print("Parameters per Sector:")
print(params_df)

print("\n\nPortfolio Weights (W):")
W_df = pd.DataFrame(W, columns=[f'Sector_{j}' for j in range(J)], index=pd.Index(range(N_firms), name='Firm'))
print(W_df)

Parameters per Sector:
            kappa    theta    sigma  lambda_benchmark
Sector_0 0.694378 0.048318 0.135817          0.259574
Sector_1 1.121571 0.049001 0.152923          0.229543
Sector_2 1.408060 0.014160 0.085078          0.076515


Portfolio Weights (W):
      Sector_0  Sector_1  Sector_2
Firm                              
0     0.333333  0.333333  0.333333
1     0.333333  0.333333  0.333333
2     0.333333  0.333333  0.333333
3     0.333333  0.333333  0.333333
4     0.333333  0.333333  0.333333
...        ...       ...       ...
995   0.333333  0.333333  0.333333
996   0.333333  0.333333  0.333333
997   0.333333  0.333333  0.333333
998   0.333333  0.333333  0.333333
999   0.333333  0.333333  0.333333

[1000 rows x 3 columns]


In [14]:
loan_l = [
    {
        "type": 'linear',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_l, Y_max_vals_l, n_defaults_l = run_simulation(params, loan_l, num_trials=NUM_TRIALS)
metrics_l = compute_loss_metrics(losses_l, alpha=0.95)


metrics_l_df = pd.DataFrame({**metrics_l}, index=['Linear'])


loss_distribution_plot(losses_l, 'Linear Loan', tag='threshold_linear')

In [15]:
metrics_l_df

,Mean,Std,VaR,ES,Excess ES,ExcessKurtosis
Linear,421.222004,75.102296,549.849555,587.677835,37.828280,0.339696


In [16]:
print(np.corrcoef(Y_max_vals_l, losses_l)[0,1])

plot_Ymax_vs_loss(Y_max_vals_l, losses_l, tag='threshold_linear')

0.8180336624005301


## Threshold = 99.5% percentile of Y_max

In [17]:
y_star_b = np.quantile(Y_max_vals_l, 0.995)
print(y_star_b)

0.5526462409551545


In [18]:
def simulate_sector_cir_model_time(kappa, theta, sigma, T, delta, W, eps, lambda_benchmark,
                              Nfirms, idiosyncratic_factor, loan_contracts, alpha_lgd, beta_lgd, y_star, rng=None):

    import warnings

    if rng is None:
        rng = np.random.default_rng()

    # ensure shapes
    J = len(theta)
    assert W.shape == (Nfirms, J)
    idiosyncratic_factor = np.asarray(idiosyncratic_factor).reshape(Nfirms,)
    lambda_benchmark = np.asarray(lambda_benchmark).reshape(J,)

    delta = np.asarray(delta)
    if delta.ndim == 1:
        delta = np.tile(delta.reshape(1, J), (Nfirms, 1))
    assert delta.shape == (Nfirms, J)

    # init
    t = 0.0
    Y_t = theta.copy().astype(float)   # sectoral intensities (J-vector), initialised at theta
    events = []
    owed = []
    marks = []
    defaulter_idio_factor = []
    sector_contributions = []
    alive = np.arange(Nfirms, dtype=int)

    # precompute some arrays for speed
    kappa = np.asarray(kappa, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    theta = np.asarray(theta, dtype=float)

    max_Y = np.sum(Y_t)

    hit = False
    tau_star = np.inf

    # event loop
    while (t < T) and (alive.size > 0):

        lambda_max = np.maximum(Y_t, lambda_benchmark)  # J-vector
        # Hepsilon = sum_i [ X_i + w_i · lambda_max ] over alive firms
        
        # We can compute per-firm value and sum
        per_firm_sys = W[alive].dot(lambda_max)   # length alive.size
        per_firm_idio = idiosyncratic_factor[alive]
        Hepsilon = per_firm_sys.sum() + per_firm_idio.sum()

        if Hepsilon <= 0:
            break

        # sample a candidate waiting time
        tau = rng.exponential(1.0 / Hepsilon)
        t_candidate = t + tau
        if t_candidate >= T:
            break

        # sample sectoral Y at t_candidate conditional on no defaults in the waiting time
        Y_proposed = cir_transition_sample_per_sector(Y_t, tau, kappa, theta, sigma, rng)

        # compute proposed per-firm intensities (scalar values) using Y_proposed
        per_firm_sys_prop = W[alive].dot(Y_proposed)
        per_firm_total_prop = per_firm_sys_prop + per_firm_idio
        lambda_proposed_total = per_firm_total_prop.sum()

        # acceptance probability: Xi / Hepsilon where Xi = lambda_proposed_total
        accept_prob = min(lambda_proposed_total / Hepsilon, 1.0)
        if rng.random() < accept_prob:
            # accept a default at t_candidate
            t = t_candidate
            events.append(t)

            # choose which alive firm defaulted
            probs = per_firm_total_prop / lambda_proposed_total
            probs = np.maximum(probs, 0.0)
            probs = probs / probs.sum()
            selected_local_idx = rng.choice(len(alive), p=probs)
            selected_firm = int(alive[selected_local_idx])

            # record idiosyncratic factor
            defaulter_idio_factor.append(float(idiosyncratic_factor[selected_firm]))

            sector_contributions.append(W[selected_firm].copy())

            # mark (loss) from contract exposure
            contract = loan_contracts[selected_firm]
            exposure = exposure_at_time(contract, T, t)
            if exposure < 0:
                warnings.warn("Exposure computed negative; clamping to zero.")
                exposure = max(exposure, 0.0)
            if t >= T and exposure > 0:
                warnings.warn("Exposure positive after maturity; forcing to zero.")
                exposure = 0.0

            owed.append(exposure)
            
            LGD = rng.beta(alpha_lgd, beta_lgd)
            mark = LGD * exposure
            # mark =  exposure
            if mark < 0:
                warnings.warn("Loss mark negative; clamping to zero.")
                mark = max(mark, 0.0)
            marks.append(mark)

            # update Y_t with contagion
            Y_t = Y_proposed + delta[selected_firm] * mark/contract["P"]

            # remove defaulted firm from alive set
            alive = np.delete(alive, selected_local_idx)

        else:
            # reject, ie no default: advance time and set Y_t = Y_proposed
            t = t_candidate
            Y_t = Y_proposed
        
        max_Y = max(max_Y, np.sum(Y_t))

        if (not hit) and (y_star is not None) and (np.sum(Y_t) >= np.quantile(y_star, 0.9)):
            hit = True
            tau_star = t

    return (np.array(events), np.array(marks), np.array(owed),
            np.array(defaulter_idio_factor), np.array(sector_contributions), max_Y, tau_star)

In [19]:
def run_simulation_time(params, loan_contracts, threshold, num_trials, seed=42):

    rng = np.random.default_rng(seed)
    eps = 1e-5
    losses = np.zeros(num_trials)
    Y_max_vals = np.zeros(num_trials)
    num_defaults = np.zeros(num_trials, dtype=int)
    fp_times = np.full(num_trials, np.inf)

    for i in range(num_trials):
        events, marks, _, _, _, max_Y, tau_star = simulate_sector_cir_model_time(params['kappa'], params['theta'], params['sigma'], 
                          params['T'], params['delta'], params['W'], 1e-5, 
                          params['lambda_benchmark'], 
                          params['Firms'], params['idio_factor'], 
                          loan_contracts,
                          alpha_lgd, beta_lgd,
                          threshold,
                          rng)
        losses[i] = np.sum(marks)
        Y_max_vals[i] = max_Y
        num_defaults[i] = len(events)

        if tau_star < np.inf:
            fp_times[i] = tau_star

    return losses, Y_max_vals, num_defaults, fp_times

In [20]:
def compute_fp_metrics(fp_times):
    prob_fp = np.mean(fp_times < T)
    finite = fp_times[np.isfinite(fp_times)]
    if len(finite) == 0:
        mean_fp = np.inf
    else:
        mean_fp = np.mean(finite)

    return {
        "Probability of First Passage": float(np.round(prob_fp, DECIMALS)), 
        "Mean First Passage Time (finite)": float(np.round(mean_fp, DECIMALS))
    }

In [21]:
def plot_fp_survival_curves(fp_dict, T, tag=""):
    """
    fp_dict: dict
        keys   = labels (e.g. 'Bullet', 'French', ...)
        values = arrays of fp_times (length = num_trials)
    """
    line_colors = [COLORS['dark'], COLORS['kde'], COLORS['green'], COLORS['purple']]
    fig, ax = plt.subplots(figsize=(6, 4))

    for idx, (label, fp_times) in enumerate(fp_dict.items()):
        finite = fp_times[np.isfinite(fp_times)]
        if len(finite) == 0:
            continue
        ts = np.sort(finite)
        n = len(fp_times)
        surv = 1.0 - np.arange(1, len(ts) + 1) / n
        c = line_colors[idx % len(line_colors)]
        m = MARKERS[idx % len(MARKERS)]
        step_x = np.concatenate([[ts[0]], ts])
        step_y = np.concatenate([[1.0], surv])
        ax.plot(step_x, step_y, drawstyle='steps-post', label=label,
                color=c, linewidth=1.5)

    ax.set_xlabel("Time")
    ax.set_ylabel(r"Survival Probability $P(\tau^\star > t)$")
    ax.set_ylim(0, 1.01)
    ax.legend(frameon=False)

    if tag:
        for fmt in ('pdf', 'png'):
            fig.savefig(IMG_DIR / f"survival_curves_{tag}.{fmt}", format=fmt)
    plt.close(fig)

## Balanced sectoral config; Lower contagion (0.01)

In [22]:
rng = np.random.default_rng(1265883927568217)

N_firms = 1000
J = 3

c = 0.01

T = 2.0

NUM_TRIALS = 2500

kappa = rng.uniform(0.5, 1.5, J)
theta = rng.uniform(0.001, 0.051, J)
sigma_inter = rng.uniform(0.0, 0.2, J)
sigma = np.minimum(np.sqrt(2*kappa*theta), sigma_inter)

idio_factor = rng.uniform(0.01, 0.03, N_firms)

feller = (2 * kappa * theta) / (sigma**2)
print("Feller condition (should be >=1):", feller)

W = generate_portfolio_weights(N_firms, J, "balanced", rng=rng)

delta = c * W

lambda_benchmark = np.array([
    lambda_max_generator(1e-4, theta[j], theta[j],
                         sigma[j], kappa[j])
    for j in range(J)
])


params = {
    'Firms': N_firms,
    'Sectors': J,
    'Global Senstivity Param': c,
    'kappa': kappa,
    'theta': theta,
    'sigma': sigma,
    'delta': delta,
    'lambda_benchmark': lambda_benchmark,
    'idio_factor': idio_factor,
    'W': W,
    'T': T
}

Feller condition (should be >=1): [3.637663 4.700201 5.508869]


In [23]:
params_df = pd.DataFrame({
    'kappa': kappa,
    'theta': theta,
    'sigma': sigma,
    'lambda_benchmark': lambda_benchmark
}, index=[f'Sector_{j}' for j in range(J)])

print("Parameters per Sector:")
print(params_df)

print("\n\nPortfolio Weights (W):")
W_df = pd.DataFrame(W, columns=[f'Sector_{j}' for j in range(J)], index=pd.Index(range(N_firms), name='Firm'))
print(W_df)

Parameters per Sector:
            kappa    theta    sigma  lambda_benchmark
Sector_0 0.694378 0.048318 0.135817          0.259574
Sector_1 1.121571 0.049001 0.152923          0.229543
Sector_2 1.408060 0.014160 0.085078          0.076515


Portfolio Weights (W):
      Sector_0  Sector_1  Sector_2
Firm                              
0     0.333333  0.333333  0.333333
1     0.333333  0.333333  0.333333
2     0.333333  0.333333  0.333333
3     0.333333  0.333333  0.333333
4     0.333333  0.333333  0.333333
...        ...       ...       ...
995   0.333333  0.333333  0.333333
996   0.333333  0.333333  0.333333
997   0.333333  0.333333  0.333333
998   0.333333  0.333333  0.333333
999   0.333333  0.333333  0.333333

[1000 rows x 3 columns]


In [24]:
loan_b = [
    {
        "type": 'bullet',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_b, Y_max_vals_b, n_defaults_b, fp_times_b = run_simulation_time(params, loan_b, y_star_b, num_trials=NUM_TRIALS)
metrics_b = compute_loss_metrics(losses_b, alpha=0.95)

fp_metrics_b = compute_fp_metrics(fp_times_b)

metrics_b_df = pd.DataFrame({**metrics_b, **fp_metrics_b }, index=['Bullet'])

loss_distribution_plot(losses_b, 'Bullet Loan', tag='balanced_bullet')

In [25]:
print(metrics_b_df)

              Mean        Std         VaR          ES  Excess ES  \
Bullet 1194.439821 201.219222 1529.323246 1629.680233 100.356987   

        ExcessKurtosis  Probability of First Passage  \
Bullet        0.076372                      0.880400   

        Mean First Passage Time (finite)  
Bullet                          1.394318  


In [26]:
print(np.corrcoef(Y_max_vals_b, losses_b)[0,1])

plot_Ymax_vs_loss(Y_max_vals_b, losses_b, tag='balanced_bullet')

0.8584075100156862


In [27]:
loan_f = [
    {
        "type": 'french',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_f, Y_max_vals_f, n_defaults_f, fp_times_f = run_simulation_time(params, loan_f, y_star_b, num_trials=NUM_TRIALS)
metrics_f = compute_loss_metrics(losses_f, alpha=0.95)

fp_metrics_f = compute_fp_metrics(fp_times_f)

metrics_f_df = pd.DataFrame({**metrics_f, **fp_metrics_f}, index=['French'])
print(metrics_f_df)

loss_distribution_plot(losses_f, 'French Loan', tag='balanced_french')

print(np.corrcoef(Y_max_vals_f, losses_f)[0,1])
plot_Ymax_vs_loss(Y_max_vals_f, losses_f, tag='balanced_french')

             Mean       Std        VaR         ES  Excess ES  ExcessKurtosis  \
French 448.768545 82.627061 591.675673 628.778214  37.102541        0.060028   

        Probability of First Passage  Mean First Passage Time (finite)  
French                      0.011600                          1.170074  
0.8366658904619713


In [28]:
loan_l = [
    {
        "type": 'linear',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_l, Y_max_vals_l, n_defaults_l, fp_times_l = run_simulation_time(params, loan_l, y_star_b, num_trials=NUM_TRIALS)
metrics_l = compute_loss_metrics(losses_l, alpha=0.95)

fp_metrics_l = compute_fp_metrics(fp_times_l)

metrics_l_df = pd.DataFrame({**metrics_l, **fp_metrics_l}, index=['Linear'])
print(metrics_l_df)

loss_distribution_plot(losses_l, 'Linear Loan', tag='balanced_linear')

print(np.corrcoef(Y_max_vals_l, losses_l)[0,1])
plot_Ymax_vs_loss(Y_max_vals_l, losses_l, tag='balanced_linear')

             Mean       Std        VaR         ES  Excess ES  ExcessKurtosis  \
Linear 421.222004 75.102296 549.849555 587.677835  37.828280        0.339696   

        Probability of First Passage  Mean First Passage Time (finite)  
Linear                      0.005200                          1.130230  
0.8180336624005301


In [29]:
loan_n = [
    {
        "type": 'negative',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_n, Y_max_vals_n, n_defaults_n, fp_times_n = run_simulation_time(params, loan_n, y_star_b, num_trials=NUM_TRIALS)
metrics_n = compute_loss_metrics(losses_n, alpha=0.95)

fp_metrics_n = compute_fp_metrics(fp_times_n)

metrics_n_df = pd.DataFrame({**metrics_n, **fp_metrics_n}, index=['Neg. Amort.'])
print(metrics_n_df)

loss_distribution_plot(losses_n, 'Negative Amortisation', tag='balanced_neg_amort')

print(np.corrcoef(Y_max_vals_n, losses_n)[0,1])
plot_Ymax_vs_loss(Y_max_vals_n, losses_n, tag='balanced_neg_amort')

                   Mean        Std         VaR          ES  Excess ES  \
Neg. Amort. 1520.019444 248.808323 1935.959243 2057.780137 121.820894   

             ExcessKurtosis  Probability of First Passage  \
Neg. Amort.        0.043292                      0.984400   

             Mean First Passage Time (finite)  
Neg. Amort.                          1.262939  
0.8764401677479994


In [30]:
metrics1 = pd.concat([metrics_b_df, metrics_f_df, metrics_l_df, metrics_n_df])
metrics1

,Mean,Std,VaR,ES,Excess ES,ExcessKurtosis,Probability of First Passage,Mean First Passage Time (finite)
Bullet,1194.439821,201.219222,1529.323246,1629.680233,100.356987,0.076372,0.880400,1.394318
French,448.768545,82.627061,591.675673,628.778214,37.102541,0.060028,0.011600,1.170074
Linear,421.222004,75.102296,549.849555,587.677835,37.828280,0.339696,0.005200,1.130230
Neg. Amort.,1520.019444,248.808323,1935.959243,2057.780137,121.820894,0.043292,0.984400,1.262939


In [31]:
fp_dict_b = {
    'Bullet': fp_times_b,
    'French': fp_times_f,
    'Linear': fp_times_l,
    'Neg. Amort.': fp_times_n
}

plot_fp_survival_curves(fp_dict_b, T=2.0, tag='balanced')

## Concentrated sectoral config; Lower contagion (0.01)

In [32]:
rng = np.random.default_rng(1265883927568217)

N_firms = 1000
J = 3

c = 0.01

T = 2.0

NUM_TRIALS = 2500

kappa = rng.uniform(0.5, 1.5, J)
theta = rng.uniform(0.001, 0.051, J)
sigma_inter = rng.uniform(0.0, 0.2, J)
sigma = np.minimum(np.sqrt(2*kappa*theta), sigma_inter)

idio_factor = rng.uniform(0.01, 0.03, N_firms)

feller = (2 * kappa * theta) / (sigma**2)
print("Feller condition (should be >=1):", feller)

W = generate_portfolio_weights(N_firms, J, "concentrated", rng=rng)

delta = c * W

lambda_benchmark = np.array([
    lambda_max_generator(1e-4, theta[j], theta[j],
                         sigma[j], kappa[j])
    for j in range(J)
])


params = {
    'Firms': N_firms,
    'Sectors': J,
    'Global Senstivity Param': c,
    'kappa': kappa,
    'theta': theta,
    'sigma': sigma,
    'delta': delta,
    'lambda_benchmark': lambda_benchmark,
    'idio_factor': idio_factor,
    'W': W,
    'T': T
}


Feller condition (should be >=1): [3.637663 4.700201 5.508869]


In [33]:
params_df = pd.DataFrame({
    'kappa': kappa,
    'theta': theta,
    'sigma': sigma,
    'lambda_benchmark': lambda_benchmark
}, index=[f'Sector_{j}' for j in range(J)])

print("Parameters per Sector:")
print(params_df)

print("\n\nPortfolio Weights (W):")
W_df = pd.DataFrame(W, columns=[f'Sector_{j}' for j in range(J)], index=pd.Index(range(N_firms), name='Firm'))
print(W_df)


Parameters per Sector:
            kappa    theta    sigma  lambda_benchmark
Sector_0 0.694378 0.048318 0.135817          0.259574
Sector_1 1.121571 0.049001 0.152923          0.229543
Sector_2 1.408060 0.014160 0.085078          0.076515


Portfolio Weights (W):
      Sector_0  Sector_1  Sector_2
Firm                              
0     0.700000  0.150000  0.150000
1     0.700000  0.150000  0.150000
2     0.700000  0.150000  0.150000
3     0.700000  0.150000  0.150000
4     0.700000  0.150000  0.150000
...        ...       ...       ...
995   0.700000  0.150000  0.150000
996   0.700000  0.150000  0.150000
997   0.700000  0.150000  0.150000
998   0.700000  0.150000  0.150000
999   0.700000  0.150000  0.150000

[1000 rows x 3 columns]


In [34]:
loan_b = [
    {
        "type": 'bullet',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_b, Y_max_vals_b, n_defaults_b, fp_times_b = run_simulation_time(params, loan_b, y_star_b, num_trials=NUM_TRIALS)
metrics_b = compute_loss_metrics(losses_b, alpha=0.95)

fp_metrics_b = compute_fp_metrics(fp_times_b)

metrics_b_df = pd.DataFrame({**metrics_b, **fp_metrics_b }, index=['Bullet'])

loss_distribution_plot(losses_b, 'Bullet Loan', tag='concentrated_bullet')

In [35]:
print(metrics_b_df)

              Mean        Std         VaR          ES  Excess ES  \
Bullet 2568.366956 347.970106 3107.853779 3233.481402 125.627623   

        ExcessKurtosis  Probability of First Passage  \
Bullet       -0.059929                      1.000000   

        Mean First Passage Time (finite)  
Bullet                          0.857073  


In [36]:
print(np.corrcoef(Y_max_vals_b, losses_b)[0,1])

plot_Ymax_vs_loss(Y_max_vals_b, losses_b, tag='concentrated_bullet')

0.859710747222827


In [37]:
loan_f = [
    {
        "type": 'french',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_f, Y_max_vals_f, n_defaults_f, fp_times_f = run_simulation_time(params, loan_f, y_star_b, num_trials=NUM_TRIALS)
metrics_f = compute_loss_metrics(losses_f, alpha=0.95)

fp_metrics_f = compute_fp_metrics(fp_times_f)

metrics_f_df = pd.DataFrame({**metrics_f, **fp_metrics_f}, index=['French'])
print(metrics_f_df)

loss_distribution_plot(losses_f, 'French Loan', tag='concentrated_french')

print(np.corrcoef(Y_max_vals_f, losses_f)[0,1])
plot_Ymax_vs_loss(Y_max_vals_f, losses_f, tag='concentrated_french')

             Mean        Std         VaR          ES  Excess ES  \
French 769.572423 159.991998 1038.912341 1118.668790  79.756449   

        ExcessKurtosis  Probability of First Passage  \
French        0.037691                      0.481600   

        Mean First Passage Time (finite)  
French                          1.073009  
0.8889126961430086


In [38]:
loan_l = [
    {
        "type": 'linear',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_l, Y_max_vals_l, n_defaults_l, fp_times_l = run_simulation_time(params, loan_l, y_star_b, num_trials=NUM_TRIALS)
metrics_l = compute_loss_metrics(losses_l, alpha=0.95)

fp_metrics_l = compute_fp_metrics(fp_times_l)

metrics_l_df = pd.DataFrame({**metrics_l, **fp_metrics_l}, index=['Linear'])
print(metrics_l_df)

loss_distribution_plot(losses_l, 'Linear Loan', tag='concentrated_linear')

print(np.corrcoef(Y_max_vals_l, losses_l)[0,1])
plot_Ymax_vs_loss(Y_max_vals_l, losses_l, tag='concentrated_linear')

             Mean        Std        VaR          ES  Excess ES  \
Linear 720.473671 153.212505 982.329157 1052.174028  69.844871   

        ExcessKurtosis  Probability of First Passage  \
Linear       -0.073840                      0.420400   

        Mean First Passage Time (finite)  
Linear                          1.087132  
0.8902127917656566


In [39]:
loan_n = [
    {
        "type": 'negative',
        "P": 10.0,
        "N": 12,
        "r": 0.12,
    }
    for i in range(N_firms)
]

losses_n, Y_max_vals_n, n_defaults_n, fp_times_n = run_simulation_time(params, loan_n, y_star_b, num_trials=NUM_TRIALS)
metrics_n = compute_loss_metrics(losses_n, alpha=0.95)

fp_metrics_n = compute_fp_metrics(fp_times_n)

metrics_n_df = pd.DataFrame({**metrics_n, **fp_metrics_n}, index=['Neg. Amort.'])
print(metrics_n_df)

loss_distribution_plot(losses_n, 'Negative Amortisation', tag='concentrated_neg_amort')

print(np.corrcoef(Y_max_vals_n, losses_n)[0,1])
plot_Ymax_vs_loss(Y_max_vals_n, losses_n, tag='concentrated_neg_amort')

                   Mean        Std         VaR          ES  Excess ES  \
Neg. Amort. 3319.355306 364.312344 3862.001021 3959.705078  97.704057   

             ExcessKurtosis  Probability of First Passage  \
Neg. Amort.        0.423734                      1.000000   

             Mean First Passage Time (finite)  
Neg. Amort.                          0.802325  
0.8314675950240311


In [40]:
metrics2 = pd.concat([metrics_b_df, metrics_f_df, metrics_l_df, metrics_n_df])
metrics2

,Mean,Std,VaR,ES,Excess ES,ExcessKurtosis,Probability of First Passage,Mean First Passage Time (finite)
Bullet,2568.366956,347.970106,3107.853779,3233.481402,125.627623,-0.059929,1.000000,0.857073
French,769.572423,159.991998,1038.912341,1118.668790,79.756449,0.037691,0.481600,1.073009
Linear,720.473671,153.212505,982.329157,1052.174028,69.844871,-0.073840,0.420400,1.087132
Neg. Amort.,3319.355306,364.312344,3862.001021,3959.705078,97.704057,0.423734,1.000000,0.802325


In [41]:
fp_dict_c = {
    'Bullet': fp_times_b,
    'French': fp_times_f,
    'Linear': fp_times_l,
    'Neg. Amort.': fp_times_n
}

plot_fp_survival_curves(fp_dict_c, T=2.0, tag='concentrated')